# 1. LangChain 

1.1 Fazer uma chamada simples ao Ollama/OpenAI via LangChain — instanciar uma LLM e receber uma resposta de texto

1.2 Criar um PromptTemplate e encadear com o LLM (prompt | llm)

1.3 Criar um agente sem nenhuma tool ... usar create_agent com lista vazia

1.4 Criar uma Tool customizada do zero com o decorator @tool

1.5 Conectar a tool ao agente e deixar ele decidir quando chamá-la

1.6 Logar uma run no MLflow usando o autolog do mlflow

In [1]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool

from dotenv import load_dotenv
load_dotenv()

True

In [7]:
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
# 1.1 Fazer uma chamada simples ao Ollama/OpenAI via LangChain — instanciar uma LLM e receber uma resposta de texto

# llm = ChatOllama(model="llama3")
resposta = llm.invoke("traduz amor para o japonês")
resposta.content

'A palavra "amor" em japonês é traduzida como "愛" (ai).'

In [9]:
# 1.2 Criar um PromptTemplate e encadear com o LLM (prompt | llm)
prompt_template = PromptTemplate.from_template("Me fale um sinônimo de {topico}. Resuma em poucas palavras. Seja claro e sucinto")

chain = prompt_template | llm
resposta = chain.invoke({"topico": "ashibarai"})
resposta.content

'"Ashibarai" é uma técnica de artes marciais que significa "rasteira".'

In [10]:
# 1.3 Criar um agente sem nenhuma tool ... usar create_agent com lista vazia
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful and precise assistant that give answers in portuguese"
)

# create_agente é baseado em LangGraph e então espera um formato estruturado de input (estado)
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the capital of New Zealand?"}
    ]
})
print(result)

{'messages': [HumanMessage(content='What is the capital of New Zealand?', additional_kwargs={}, response_metadata={}, id='c47a7d29-0b20-4fdc-8c5c-bd6b0a0b1154'), AIMessage(content='A capital da Nova Zelândia é Wellington.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 32, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_303cbe1edb', 'id': 'chatcmpl-DZc02R2X7CTleBJlw7PrRCI7ims6C', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dd419-5515-7cb2-833d-1a61959d7fdd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 9, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_re

In [16]:
# 1.4 Criar uma Tool customizada do zero com o decorator @tool
@tool
def simple_calculator(num1: float, num2: float, operator: str) -> str:
    """Perform a basic arithmetic operation between two numbers.

    Supported operators:
    - '+' for addition
    - '-' for subtraction
    - '*' for multiplication
    - '/' for division

    Returns the result as a string.

    Args: 
        num1: The first number in the calculation
        num2: The second number in the calculation
        operator: The operator to use in the calculation
    """

    # Perform the calculation based on the operator
    if operator == "+":
        return f"{num1 + num2}"
    elif operator == "-":
        return f"{num1 - num2}"
    elif operator == "*":
        return f"{num1 * num2}"
    elif operator == "/":
        if num2 == 0:
            return "Cannot divide by zero."
        return f"{num1 / num2}"
    
    return "Unsupported operator."

In [19]:
# 1.5 Conectar a tool ao agente e deixar ele decidir quando chamá-la
agent = create_agent(
    model=llm,
    tools=[simple_calculator],
    system_prompt="You are a helpful and precise assistant for solving math problems. You should return" \
    "the answer of the question. Use tools if necessary. "
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "I would like to know the result of 2*254"}
    ]
})
print(result)

{'messages': [HumanMessage(content='I would like to know the result of 2*254', additional_kwargs={}, response_metadata={}, id='cef88c28-0dfa-4ccf-a084-b935b7058fab'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 164, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_bac4e0cb77', 'id': 'chatcmpl-DZcT2cbsngoV7ziAS2MmxKpgK0RlD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd434-c4b6-78d1-aa20-0613089409e6-0', tool_calls=[{'name': 'simple_calculator', 'args': {'num1': 2, 'num2': 254, 'operator': '*'}, 'id': 'call_pakjp3Y6gKln8Znmkh1A8jX2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

Trace(trace_id=tr-321c8e334f53ba1139ea049da3794971)

In [ ]:
# 1.6 Logar uma run no MLflow usando o autolog do mlflow 
import mlflow

# Enabling tracing for LangGraph (LangChain)
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
#mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("LangGraph")

# 1.5 Conectar a tool ao agente e deixar ele decidir quando chamá-la
agent = create_agent(
    model=llm,
    tools=[simple_calculator],
    system_prompt="You are a helpful and precise assistant for solving math problems. You should return" \
    "the answer of the question. Use tools if necessary. "
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "I would like to know the result of 2*254"}
    ]
})

# mlflow server \
#   --host 0.0.0.0 \
#   --port 5000 \
#   --dev \
#   --cors-allowed-origins "*"
# http://172.25.117.120:5000 linux -> windows

Trace(trace_id=tr-e49e16b438dfd0848c770efccb8d108d)